## EN: 1. Scope and safe configuration

Use a new demo Lakehouse named Mining_Close_Demo, attached as this notebook's default Lakehouse. This is synthetic expense/capex for June-August 2026, not a full general ledger. The source context date is 9 September 2026, the close cutoff is 31 August, and bank evidence extends to 1 September. Runtime UTC is separate. The provided staged normalization is an honest handoff, not a claim that this notebook implements every ERP transformation. Bounded driver CSV ingestion is deliberate for this small kit. All money uses Decimal. Float-labelled production inputs are read as exact decimal strings. Failed schema, hash, precision or baseline assertions stop the run; do not bypass them.

## ID: 1. Cakupan dan konfigurasi aman

Gunakan Lakehouse demo baru bernama Mining_Close_Demo, dilampirkan sebagai Lakehouse default notebook ini. Data sintetis mencakup beban/capex Juni-Agustus 2026, bukan buku besar lengkap. Tanggal konteks sumber adalah 9 September 2026, batas tutup 31 Agustus, dan bukti bank sampai 1 September. UTC proses dicatat terpisah. Normalisasi staging yang disediakan adalah serah terima yang jujur, bukan klaim bahwa notebook ini melakukan seluruh transformasi ERP. Pembacaan CSV pada driver dibatasi untuk kit kecil ini. Semua uang memakai Decimal. Input produksi berlabel float dibaca sebagai string desimal persis. Assertion skema, hash, presisi atau baseline yang gagal menghentikan proses; jangan dilewati.


In [ ]:
import csv
import hashlib
import json
from collections import Counter
from datetime import date, datetime, timezone
from decimal import Decimal, ROUND_HALF_UP, localcontext
from pathlib import Path
from functools import reduce

import pandas as pd
from pyspark.sql import functions as F, types as T

# These are Fabric notebook mount paths, not Windows workstation paths.
INPUT_DIR = Path("/lakehouse/default/Files/input-data")
EXPORT_DIR = Path("/lakehouse/default/Files/gold_export")
MAX_INPUT_ROWS = 20000
MAX_INPUT_BYTES = 8 * 1024 * 1024
MAX_EXPORT_ROWS = 20000
RUN_UTC = datetime.now(timezone.utc).isoformat()
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.decimalOperations.allowPrecisionLoss", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")

contract = json.loads((INPUT_DIR / "input_schema.json").read_text(encoding="utf-8"))
SOURCE_AS_OF = contract["source_as_of_date"]
CLOSE_CUTOFF = contract["close_cutoff_date"]
assert contract["fictional"] is True
assert SOURCE_AS_OF >= CLOSE_CUTOFF
assert contract["source_ledger_scope"] == "Expense/capex only; June-August 2026; not a full general ledger"
assert contract["document_type_assumption"] == "All AP rows are Invoice; supplier credits are CreditNote"

bronze = {}
silver = {}
local_rows = {}

def exact_decimal(value, precision, scale):
    number = Decimal(value)
    assert number.is_finite(), "Non-finite numeric input"
    with localcontext() as context:
        context.prec = 60
        fixed = number.quantize(Decimal(1).scaleb(-scale))
    assert fixed == number, f"Precision would be lost: {value}"
    assert abs(number) < Decimal(10) ** (precision - scale), "Decimal overflow"
    return fixed

def decode(value, kind):
    if value is None:
        return None
    if kind.startswith("decimal"):
        precision, scale = map(int, kind[8:-1].split(","))
        return exact_decimal(value, precision, scale)
    if kind == "date":
        return date.fromisoformat(value)
    if kind == "integer":
        assert Decimal(value) == int(value)
        return int(value)
    return value

def dtype(kind):
    if kind.startswith("decimal"):
        return T.DecimalType(*map(int, kind[8:-1].split(",")))
    return {"date": T.DateType(), "integer": T.LongType(), "string": T.StringType()}[kind]

def unique(frame, columns, label):
    assert frame.groupBy(*columns).count().filter("count > 1").count() == 0, label
    assert frame.filter(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in columns])).count() == 0, label

def no_rows(frame, label):
    assert frame.limit(1).count() == 0, label

def register(name, frame):
    frame.createOrReplaceTempView(name)
    return frame

GOLD_COLUMNS = {'gold_normalized_ledger': ['SourceRowID', 'Site', 'Period', 'PostingDate', 'Account', 'AccountName', 'VendorID', 'Currency', 'AmountLocal', 'AmountUSD', 'CostCentre', 'DocumentRef', 'NormalizationStatus', 'NativeSourceKey', 'RawSourceFile', 'PayloadHash', 'IsRepeatedImport', 'FXRate', 'CalculatedUSD', 'IsUnmappedAccount', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_control_reconciliation': ['Site', 'Period', 'Account', 'LedgerPresent', 'ControlPresent', 'OccurrenceRows', 'NullUSDRows', 'NullAccountRows', 'UnmappedAccountRows', 'NullVendorRows', 'RepeatedImportRows', 'KnownUSD_AllOccurrences', 'KnownUSD_ReplayExcluded', 'ControlUSD', 'DifferenceUSD_AllOccurrences', 'DifferenceUSD_ReplayExcluded', 'ControlStatus', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_ap_review': ['InvoiceID', 'Site', 'VendorID', 'VendorName', 'InvoiceNo', 'InvoiceDate', 'DueDate', 'POID', 'Currency', 'GrossAmount', 'TaxAmount', 'NetAmount', 'PaymentRef', 'Status', 'DocumentType', 'VendorIDKey', 'InvoiceNoKey', 'CurrencyKey', 'DuplicateGroupRows', 'PaidRowsInGroup', 'OpenRowsInGroup', 'DuplicateCandidate', 'Period', 'FXRate', 'GrossUSD', 'CandidateOpenDuplicateUSD', 'POVendorID', 'OrderedAmount', 'ApprovedAmount', 'ReceivedAmount', 'ReceiptRows', 'PriceDifferenceLocal', 'ReceiptDifferenceLocal', 'MissingReceipt', 'MissingPO', 'POVendorMismatch', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_bank_cash_reconciliation': ['Ref', 'Currency', 'BankSites', 'CashSites', 'BankRows', 'CashRows', 'BankPostCutoffRows', 'CashPostCutoffRows', 'BankAllDates', 'CashAllDates', 'BankAtClose', 'CashAtClose', 'DifferenceAllDates', 'DifferenceAtClose', 'MatchStatus', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_supplier_credits': ['StatementLineID', 'StatementDate', 'Site', 'VendorID', 'InvoiceNo', 'Currency', 'SignedAmount', 'InvoiceID', 'CreditNoteID', 'EntryType', 'EvidenceRef', 'DocumentType', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_budget_summary': ['Site', 'Period', 'Account', 'BudgetID', 'AccountName', 'DriverUnit', 'BudgetVolume', 'BudgetRateUSD', 'BudgetUSD', 'BudgetActualUSD', 'SuppliedVarianceUSD', 'Assessment', 'ControlRef', 'KnownUSD_AllOccurrences', 'KnownUSD_ReplayExcluded', 'ControlUSD', 'VarianceUSD', 'BudgetMinusControlUSD', 'ControlStatus', 'NullUSDRows', 'UnmappedAccountRows', 'RepeatedImportRows', 'BudgetStatus', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_operations_summary': ['Site', 'Period', 'ProductionRows', 'PlannedOreTonnes', 'ActualOreTonnes', 'OreTonnesVariance', 'RecoveredGoldOz', 'PlannedGoldOz', 'GoldOzVariance', 'FirstProductionDate', 'LastProductionDate', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_exception_register': ['ExceptionID', 'Domain', 'RecordID', 'Site', 'Period', 'Account', 'Currency', 'IssueType', 'ReviewAmount', 'AmountBasis', 'EvidenceRef', 'SourceAsOfDate', 'CloseCutoffDate', 'RunUTC'], 'gold_source_metadata': ['SourceFile', 'InputRows', 'SHA256', 'SourceAsOfDate', 'CloseCutoffDate', 'LedgerScope', 'ControlAssumption', 'SnapshotStatus', 'RunUTC']}


## EN: 2. Bronze raw evidence and typed silver

Upload the extracted input-data folder into Lakehouse Files before running. All 15 CSV files are ingested without losing columns. Three raw_ERP files are preserved as string bronze Delta tables, with all 1,004 occurrences per ERP. The canonical_ledger_raw CSV is already normalized by the data-generation preparation stage. Silver assigns explicit date/Decimal types but retains all 3,012 rows and all exceptions. Unique SourceRowID identifies an import occurrence, not a transaction to deduplicate. This fixed snapshot checks SHA256, exact ordered headers, row widths, nullability, precision, row counts and declared keys. Tables with this kit's bronze_/silver_ names are overwritten in the demo Lakehouse.

## ID: 2. Bukti mentah bronze dan silver bertipe

Unggah folder input-data hasil ekstraksi ke Lakehouse Files sebelum menjalankan. Seluruh 15 CSV dibaca tanpa kehilangan kolom. Tiga file raw_ERP dipertahankan sebagai tabel Delta bronze string, dengan 1.004 kejadian per ERP. CSV canonical_ledger_raw telah dinormalisasi pada tahap persiapan data sintetis. Silver menetapkan tipe tanggal/Decimal eksplisit tetapi mempertahankan 3.012 baris dan seluruh pengecualian. SourceRowID unik mengidentifikasi kejadian impor, bukan transaksi untuk dideduplikasi. Snapshot tetap ini memeriksa SHA256, urutan header persis, lebar baris, nullability, presisi, jumlah baris dan kunci. Tabel bronze_/silver_ milik kit ditimpa pada Lakehouse demo.


In [ ]:
for filename, spec in contract["csv_files"].items():
    path = INPUT_DIR / filename
    assert path.stat().st_size <= MAX_INPUT_BYTES, f"Driver bound exceeded: {filename}"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == spec["sha256"], f"Changed input: {filename}"
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.reader(stream)
        header = next(reader)
        assert header == spec["columns"], f"Header mismatch: {filename}"
        records = []
        for line_no, row in enumerate(reader, 2):
            assert len(row) == len(header), f"Malformed row: {filename}:{line_no}"
            assert len(records) < MAX_INPUT_ROWS, f"Row bound exceeded: {filename}"
            records.append(tuple(None if value == "" else value for value in row))
    assert len(records) == spec["rows"], f"Row count mismatch: {filename}"
    stem = Path(filename).stem.lower()
    raw_schema = T.StructType([T.StructField(c, T.StringType(), True) for c in header])
    bronze[stem] = spark.createDataFrame(records, raw_schema)
    bronze[stem].write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_" + stem)
    typed = []
    for row in records:
        values = []
        for col, value in zip(header, row):
            definition = spec["column_types"][col]
            assert value is not None or definition["nullable"], f"Required value missing: {filename}.{col}"
            values.append(decode(value, definition["type"]))
        typed.append(tuple(values))
    schema = T.StructType([
        T.StructField(col, dtype(spec["column_types"][col]["type"]), spec["column_types"][col]["nullable"])
        for col in header
    ])
    frame = spark.createDataFrame(typed, schema)
    assert frame.columns == header
    assert frame.count() == len(records)
    silver[stem] = register("s_" + stem, frame)
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_" + stem)
    local_rows[stem] = [dict(zip(header, row)) for row in typed]

assert sum(bronze[k].count() for k in ["raw_erp_north", "raw_erp_east", "raw_erp_central"]) == 3012
for name, key in {
    "canonical_ledger_raw": ["SourceRowID"],
    "ap_invoices": ["InvoiceID"],
    "po_lines": ["POID", "Currency"],
    "goods_receipts": ["GRNID"],
    "bank_lines": ["BankLineID"],
    "cash_book": ["CashLineID"],
    "vendor_statements": ["StatementLineID"],
    "ledger_control": ["Site", "Period", "Account"],
    "fx_rates": ["Period", "Currency"],
    "budget": ["Site", "Period", "Account"],
    "operations": ["ProductionID"],
}.items():
    unique(silver[name], key, name)


## EN: 3. Preserve occurrence lineage and exceptions

Join each staged occurrence to its raw ERP filename and native source key, one-to-one. Repeated native keys must have identical staged business payload and an explicit RepeatedImport label on the extra occurrence. Retain all 12 repeats. Retain five UnknownAccount, three MissingFX and six AmbiguousVendor rows. FX validation uses only provided monthly rates and half-up cents; no treasury-only rate or hidden true account is inserted. CalculatedUSD remains null for missing FX. Known USD is a partial amount, not a complete balance.

## ID: 3. Pertahankan jejak kejadian dan pengecualian

Hubungkan setiap kejadian staging ke nama file ERP mentah dan kunci sumber asli secara satu-ke-satu. Kunci asli berulang harus memiliki isi bisnis staging identik dan label RepeatedImport eksplisit pada kejadian tambahan. Pertahankan 12 impor ulang. Pertahankan lima UnknownAccount, tiga MissingFX dan enam AmbiguousVendor. Validasi FX hanya memakai kurs bulanan yang disediakan dan pembulatan half-up sen; kurs khusus treasury atau akun benar tersembunyi tidak dimasukkan. CalculatedUSD tetap null untuk FX kosong. USD diketahui adalah nilai parsial, bukan saldo lengkap.


In [ ]:
identity = reduce(lambda a, b: a.unionByName(b), [
    bronze[stem].select("SourceRowID", F.col(key).alias("NativeSourceKey"), F.lit(filename).alias("RawSourceFile"))
    for stem, key, filename in [
        ("raw_erp_north", "ImportLineKey", "raw_ERP_North.csv"),
        ("raw_erp_east", "KunciAsli", "raw_ERP_East.csv"),
        ("raw_erp_central", "NativeKey", "raw_ERP_Central.csv"),
    ]
])
unique(identity, ["SourceRowID"], "Raw occurrence IDs")
stage = silver["canonical_ledger_raw"]
no_rows(stage.join(identity, "SourceRowID", "left_anti"), "Stage occurrence absent from raw")
no_rows(identity.join(stage, "SourceRowID", "left_anti"), "Raw occurrence absent from stage")
ledger = stage.join(identity, "SourceRowID", "left")
payload_columns = [c for c in stage.columns if c not in ["SourceRowID", "NormalizationStatus"]]
ledger = ledger.withColumn("PayloadHash", F.sha2(F.to_json(F.struct(*payload_columns)), 256))
replay_groups = ledger.groupBy("RawSourceFile", "NativeSourceKey").agg(
    F.count("*").alias("OccurrenceCount"),
    F.countDistinct("PayloadHash").alias("PayloadCount"),
    F.sum(F.when(F.col("NormalizationStatus") == "RepeatedImport", 1).otherwise(0)).alias("ReplayCount")
)
no_rows(replay_groups.filter("(OccurrenceCount > 1 AND (PayloadCount <> 1 OR ReplayCount <> OccurrenceCount - 1)) OR (OccurrenceCount = 1 AND ReplayCount <> 0)"), "Replay classification not supported by payload")
assert replay_groups.filter("OccurrenceCount > 1").count() == 12
ledger = ledger.withColumn("IsRepeatedImport", F.col("NormalizationStatus") == "RepeatedImport")
rates = silver["fx_rates"].withColumnRenamed("USD_per_unit", "FXRate")
ledger = ledger.join(rates, ["Period", "Currency"], "left")

@F.udf(T.DecimalType(20, 2))
def translate_usd(amount, rate):
    if amount is None or rate is None:
        return None
    with localcontext() as context:
        context.prec = 60
        return (amount * rate).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

ledger = ledger.withColumn("CalculatedUSD", translate_usd("AmountLocal", "FXRate"))
ledger = ledger.withColumn("IsUnmappedAccount", F.col("NormalizationStatus") == "UnknownAccount")
no_rows(ledger.filter(~F.col("CalculatedUSD").eqNullSafe(F.col("AmountUSD"))), "Staged USD does not agree with provided rates")
no_rows(ledger.filter(F.col("FXRate").isNull() & (F.col("NormalizationStatus") != "MissingFX")), "Missing FX not labelled")
assert ledger.count() == 3012
assert ledger.select("SourceRowID").distinct().count() == 3012
assert {r["NormalizationStatus"]: r["count"] for r in ledger.groupBy("NormalizationStatus").count().collect()} == {
    "Ready": 2986, "RepeatedImport": 12, "UnknownAccount": 5, "MissingFX": 3, "AmbiguousVendor": 6
}
register("ledger", ledger)
gold = {"gold_normalized_ledger": ledger}


## EN: 4. Full outer control reconciliation

Aggregate by Site, Period and Account, then FULL OUTER join with null-safe keys. Preserve ledger-only, control-only, null and unmapped groups. Show row counts, unvalued USD, unmapped accounts, missing vendors and repeats alongside amounts. Do not coalesce an absent control or unvalued amount to zero. The controller certificate is independently simulated from a pre-import blueprint, true coding and treasury-only FX, excludes replays and includes three control-only adjustments: S01/August/6130 +12,500; S02/August/6110 -3,400; S03/August/1540 +8,250 USD. It is not audited independence. This notebook never derives ControlUSD from the normalized ledger. All-occurrence known USD is 26,805,066.34 versus control 26,741,850.07. The aggregate difference is +63,216.27. A replay-excluded diagnostic is -33,761.08, not a corrected ledger. Do not sum nullable row differences as the aggregate gap, because unmatched groups have no row difference.

## ID: 4. Rekonsiliasi kontrol full outer

Agregasikan menurut Site, Period dan Account, lalu FULL OUTER join dengan kunci aman-null. Pertahankan kelompok hanya-ledger, hanya-kontrol, null dan tidak terpetakan. Tampilkan jumlah baris, USD tanpa nilai, akun tidak terpetakan, pemasok kosong dan impor ulang bersama nilainya. Jangan ubah kontrol yang tidak tersedia atau nilai yang tidak diketahui menjadi nol. Sertifikat controller disimulasikan independen dari blueprint sebelum impor, kode benar dan FX khusus treasury, tanpa impor ulang serta dengan tiga penyesuaian khusus kontrol: S01/Agustus/6130 +12.500; S02/Agustus/6110 -3.400; S03/Agustus/1540 +8.250 USD. Ini bukan independensi audit. Notebook tidak menghitung ControlUSD dari ledger ternormalisasi. USD diketahui seluruh kejadian adalah 26.805.066,34 versus kontrol 26.741.850,07. Selisih agregat +63.216,27. Diagnostik tanpa impor ulang adalah -33.761,08, bukan ledger yang dikoreksi. Jangan menjumlahkan selisih baris nullable sebagai selisih agregat, karena kelompok tanpa pasangan tidak memiliki selisih baris.


In [ ]:
control_sql = """
WITH actual AS (
 SELECT Site, Period, Account,
   COUNT(*) AS OccurrenceRows,
   SUM(CASE WHEN AmountUSD IS NULL THEN 1 ELSE 0 END) AS NullUSDRows,
   SUM(CASE WHEN Account IS NULL THEN 1 ELSE 0 END) AS NullAccountRows,
   SUM(CASE WHEN IsUnmappedAccount THEN 1 ELSE 0 END) AS UnmappedAccountRows,
   SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS NullVendorRows,
   SUM(CASE WHEN IsRepeatedImport THEN 1 ELSE 0 END) AS RepeatedImportRows,
   SUM(AmountUSD) AS KnownUSD_AllOccurrences,
   SUM(CASE WHEN NOT IsRepeatedImport THEN AmountUSD ELSE CAST(0 AS DECIMAL(20,2)) END) AS KnownUSD_ReplayExcluded
 FROM ledger GROUP BY Site, Period, Account
)
SELECT COALESCE(a.Site,c.Site) AS Site, COALESCE(a.Period,c.Period) AS Period,
 COALESCE(a.Account,c.Account) AS Account,
 a.OccurrenceRows IS NOT NULL AS LedgerPresent, c.ControlUSD IS NOT NULL AS ControlPresent,
 COALESCE(a.OccurrenceRows,0) AS OccurrenceRows, COALESCE(a.NullUSDRows,0) AS NullUSDRows,
 COALESCE(a.NullAccountRows,0) AS NullAccountRows, COALESCE(a.UnmappedAccountRows,0) AS UnmappedAccountRows,
 COALESCE(a.NullVendorRows,0) AS NullVendorRows, COALESCE(a.RepeatedImportRows,0) AS RepeatedImportRows,
 a.KnownUSD_AllOccurrences, a.KnownUSD_ReplayExcluded, c.ControlUSD,
 a.KnownUSD_AllOccurrences-c.ControlUSD AS DifferenceUSD_AllOccurrences,
 a.KnownUSD_ReplayExcluded-c.ControlUSD AS DifferenceUSD_ReplayExcluded,
 CASE WHEN a.OccurrenceRows IS NULL THEN 'ControlOnly'
      WHEN c.ControlUSD IS NULL THEN 'NoControl'
      WHEN a.NullUSDRows > 0 OR a.NullAccountRows > 0 OR a.UnmappedAccountRows > 0 OR a.NullVendorRows > 0 OR a.RepeatedImportRows > 0 THEN 'ReviewRequired'
      WHEN a.KnownUSD_AllOccurrences <> c.ControlUSD THEN 'AmountDifference'
      ELSE 'Matched' END AS ControlStatus
FROM actual a FULL OUTER JOIN s_ledger_control c
 ON a.Site <=> c.Site AND a.Period <=> c.Period AND a.Account <=> c.Account
"""
control = register("control_reconciliation", spark.sql(control_sql))
assert control.agg(F.sum("OccurrenceRows")).first()[0] == 3012
assert control.count() == 95
assert control.filter("NOT ControlPresent").count() == 5
assert control.agg(F.sum("NullUSDRows")).first()[0] == 3
assert control.agg(F.sum("UnmappedAccountRows")).first()[0] == 5
assert control.agg(F.sum("NullVendorRows")).first()[0] == 6
assert control.agg(F.sum("KnownUSD_AllOccurrences")).first()[0] == Decimal("26805066.34")
assert control.agg(F.sum("ControlUSD")).first()[0] == Decimal("26741850.07")
gold["gold_control_reconciliation"] = control


## EN: 5. Invoice candidates and receipt-safe joins

The AP source has no document-type column. Its explicit contract is Invoice only; supplier CreditNote rows remain separate. Normalize VendorID and Currency by trim plus uppercase. InvoiceNoKey is uppercase with whitespace and hyphens removed, matching the supplied INV-0001 / ' inv 0001 ' variants; retain the original InvoiceNo and all other punctuation. This is a declared candidate-detection rule, not permission to merge real invoices. Use exact VendorIDKey + InvoiceNoKey + CurrencyKey + DocumentType, not invoice number alone or fuzzy vendor names. Twenty-four rows in 12 groups are candidates. USD 23,849.46 is the open-row exposure only where one paid and one open invoice form a two-row pair, not realized savings. Aggregate goods receipts by POID and Currency before joining to AP; validate PO uniqueness. Twenty split receipts do not multiply the 1,092 AP rows. Preserve ten missing receipt rows and 12 price differences, each in its local currency.

## ID: 5. Kandidat faktur dan join penerimaan aman

Sumber AP tidak memiliki kolom jenis dokumen. Kontrak eksplisitnya hanya Invoice; baris CreditNote pemasok tetap terpisah. Normalisasikan VendorID dan Currency dengan trim serta huruf besar. InvoiceNoKey memakai huruf besar serta menghapus spasi dan tanda hubung, sesuai variasi INV-0001 / ' inv 0001 ' yang disediakan; pertahankan InvoiceNo asli dan tanda baca lainnya. Ini aturan deteksi kandidat yang dinyatakan, bukan izin menggabungkan faktur nyata. Gunakan VendorIDKey + InvoiceNoKey + CurrencyKey + DocumentType secara persis, bukan nomor faktur saja atau nama pemasok fuzzy. Dua puluh empat baris dalam 12 kelompok adalah kandidat. USD 23.849,46 adalah eksposur baris terbuka hanya ketika satu faktur dibayar dan satu terbuka membentuk pasangan dua baris, bukan penghematan terealisasi. Agregasikan penerimaan barang menurut POID dan Currency sebelum join ke AP; validasi keunikan PO. Dua puluh penerimaan terpisah tidak menggandakan 1.092 baris AP. Pertahankan sepuluh baris tanpa penerimaan dan 12 selisih harga, masing-masing dalam mata uang lokal.


In [ ]:
ap = silver["ap_invoices"]
no_rows(ap.filter("GrossAmount < 0"), "Negative AP row requires an explicit document-type contract")
ap = ap.withColumn("DocumentType", F.lit("Invoice"))
for source, target in [("VendorID", "VendorIDKey"), ("InvoiceNo", "InvoiceNoKey"), ("Currency", "CurrencyKey")]:
    key = F.upper(F.trim(F.col(source)))
    if source == "InvoiceNo":
        key = F.regexp_replace(key, r"[\s-]+", "")
    ap = ap.withColumn(target, key)
    no_rows(ap.filter(F.col(target).isNull() | (F.col(target) == "")), "Missing AP business key")
duplicate_keys = ["VendorIDKey", "InvoiceNoKey", "CurrencyKey", "DocumentType"]
dupes = ap.groupBy(*duplicate_keys).agg(
    F.count("*").alias("DuplicateGroupRows"),
    F.sum(F.when(F.col("Status") == "Paid", 1).otherwise(0)).alias("PaidRowsInGroup"),
    F.sum(F.when(F.col("Status") == "Open", 1).otherwise(0)).alias("OpenRowsInGroup")
)
ap = ap.join(dupes, duplicate_keys, "left")
ap = ap.withColumn("DuplicateCandidate", F.col("DuplicateGroupRows") > 1)
ap = ap.withColumn("Period", F.date_format("InvoiceDate", "yyyy-MM"))
ap = ap.join(rates, ["Period", "Currency"], "left")
ap = ap.withColumn("GrossUSD", translate_usd("GrossAmount", "FXRate"))
ap = ap.withColumn("CandidateOpenDuplicateUSD", F.when(
    (F.col("DuplicateGroupRows") == 2) & (F.col("PaidRowsInGroup") == 1) &
    (F.col("OpenRowsInGroup") == 1) & (F.col("Status") == "Open"), F.col("GrossUSD")))
receipts = silver["goods_receipts"].groupBy("POID", "Currency").agg(
    F.sum("ReceivedAmount").alias("ReceivedAmount"), F.count("*").alias("ReceiptRows"))
po = silver["po_lines"].select("POID", "Currency", F.col("VendorID").alias("POVendorID"), "OrderedAmount", "ApprovedAmount")
ap = ap.join(po, ["POID", "Currency"], "left").join(receipts, ["POID", "Currency"], "left")
ap = ap.withColumn("PriceDifferenceLocal", F.col("NetAmount") - F.col("ApprovedAmount"))
ap = ap.withColumn("ReceiptDifferenceLocal", F.col("NetAmount") - F.col("ReceivedAmount"))
ap = ap.withColumn("MissingReceipt", F.col("ReceiptRows").isNull())
ap = ap.withColumn("MissingPO", F.col("ApprovedAmount").isNull())
ap = ap.withColumn("POVendorMismatch", ~F.col("VendorID").eqNullSafe(F.col("POVendorID")))
assert ap.count() == 1092
assert ap.filter("DuplicateCandidate").count() == 24
assert ap.filter("MissingReceipt").count() == 10
assert ap.filter("PriceDifferenceLocal <> 0").count() == 12
assert receipts.filter("ReceiptRows > 1").count() == 20
assert ap.agg(F.sum("CandidateOpenDuplicateUSD")).first()[0] == Decimal("23849.46")
register("ap_review", ap)
gold["gold_ap_review"] = ap


## EN: 6. Currency-safe cash matching and separate credits

Validate the supplied 12 batch allocations, matching each cash amount and aggregate bank amount before mapping a cash reference to the batch bank reference. Then aggregate both sides by Ref and Currency and use FULL OUTER join. Preserve all 535 bank and 544 cash rows. Separate all-date amounts from the 31 August cutoff. Four subsequent bank settlements are timing differences, eight cash-only and seven bank-only references remain for review. No timing item becomes an automatic journal. Eight supplier credits are separate document types, not duplicate invoices. No currency mixing or invented cash savings.

## ID: 6. Pencocokan kas aman-mata-uang dan kredit terpisah

Validasi 12 alokasi batch yang disediakan, cocokkan setiap nilai kas dan total bank sebelum memetakan referensi kas ke referensi batch bank. Lalu agregasikan kedua sisi menurut Ref dan Currency serta gunakan FULL OUTER join. Pertahankan 535 baris bank dan 544 baris kas. Pisahkan nilai semua tanggal dari batas 31 Agustus. Empat pelunasan bank berikutnya adalah selisih waktu; delapan referensi hanya-kas dan tujuh hanya-bank tetap untuk ditinjau. Isu waktu tidak menjadi jurnal otomatis. Delapan kredit pemasok adalah jenis dokumen terpisah, bukan faktur duplikat. Tidak ada pencampuran mata uang atau penghematan kas rekaan.


In [ ]:
bank = silver["bank_lines"]
cash = silver["cash_book"]
batch = silver["bank_batch"]
unique(batch, ["CashLineID"], "A cash row cannot be allocated more than once in this small contract")
no_rows(batch.join(bank.select("BankLineID"), "BankLineID", "left_anti"), "Batch bank row missing")
no_rows(batch.join(cash.select("CashLineID"), "CashLineID", "left_anti"), "Batch cash row missing")
allocation_check = batch.alias("x").join(cash.alias("c"), F.col("x.CashLineID") == F.col("c.CashLineID"))
no_rows(allocation_check.filter(
    (F.col("x.Currency") != F.col("c.Currency")) |
    (F.col("x.Site") != F.col("c.Site")) |
    (F.col("x.AllocatedAmount") != F.col("c.SignedAmount"))), "Batch allocation does not equal cash line")
batch_totals = batch.groupBy("BankLineID", "Currency").agg(F.sum("AllocatedAmount").alias("AllocatedAmount"))
batch_check = batch_totals.alias("x").join(bank.alias("b"), F.col("x.BankLineID") == F.col("b.BankLineID"))
no_rows(batch_check.filter(
    (F.col("x.Currency") != F.col("b.Currency")) |
    (F.col("x.AllocatedAmount") != F.col("b.SignedAmount"))), "Batch total does not equal bank line")
mapping = batch.join(bank.select("BankLineID", F.col("Ref").alias("BatchRef")), "BankLineID").select("CashLineID", "BatchRef")
cash = cash.join(mapping, "CashLineID", "left").withColumn("MatchRef", F.coalesce("BatchRef", "Ref"))
assert cash.count() == 544
register("cash_matched_reference", cash)
bank_sql = f"""
WITH b AS (
 SELECT Ref, Currency, COUNT(*) AS BankRows, SUM(SignedAmount) AS BankAllDates,
 SUM(CASE WHEN Date <= DATE '{CLOSE_CUTOFF}' THEN SignedAmount ELSE CAST(0 AS DECIMAL(20,2)) END) AS BankAtClose,
 SUM(CASE WHEN Date > DATE '{CLOSE_CUTOFF}' THEN 1 ELSE 0 END) AS BankPostCutoffRows,
 CONCAT_WS(',', SORT_ARRAY(COLLECT_SET(Site))) AS BankSites
 FROM s_bank_lines GROUP BY Ref, Currency
), c AS (
 SELECT MatchRef AS Ref, Currency, COUNT(*) AS CashRows, SUM(SignedAmount) AS CashAllDates,
 SUM(CASE WHEN Date <= DATE '{CLOSE_CUTOFF}' THEN SignedAmount ELSE CAST(0 AS DECIMAL(20,2)) END) AS CashAtClose,
 SUM(CASE WHEN Date > DATE '{CLOSE_CUTOFF}' THEN 1 ELSE 0 END) AS CashPostCutoffRows,
 CONCAT_WS(',', SORT_ARRAY(COLLECT_SET(Site))) AS CashSites
 FROM cash_matched_reference GROUP BY MatchRef, Currency
)
SELECT COALESCE(b.Ref,c.Ref) AS Ref, COALESCE(b.Currency,c.Currency) AS Currency,
 b.BankSites,c.CashSites, COALESCE(b.BankRows,0) AS BankRows,COALESCE(c.CashRows,0) AS CashRows,
 COALESCE(b.BankPostCutoffRows,0) AS BankPostCutoffRows,COALESCE(c.CashPostCutoffRows,0) AS CashPostCutoffRows,
 b.BankAllDates,c.CashAllDates,b.BankAtClose,c.CashAtClose,
 b.BankAllDates-c.CashAllDates AS DifferenceAllDates,
 b.BankAtClose-c.CashAtClose AS DifferenceAtClose,
 CASE WHEN b.BankRows IS NULL THEN 'UnmatchedCash'
      WHEN c.CashRows IS NULL THEN 'UnmatchedBank'
      WHEN b.BankAllDates <> c.CashAllDates THEN 'AmountDifference'
      WHEN b.BankAtClose <> c.CashAtClose THEN 'TimingDifference'
      ELSE 'Matched' END AS MatchStatus
FROM b FULL OUTER JOIN c ON b.Ref <=> c.Ref AND b.Currency <=> c.Currency
"""
bank_recon = register("bank_cash_reconciliation", spark.sql(bank_sql))
assert bank_recon.agg(F.sum("BankRows"), F.sum("CashRows")).first() == (535, 544)
assert bank_recon.filter("MatchStatus = 'TimingDifference'").count() == 4
assert bank_recon.filter("MatchStatus = 'UnmatchedCash'").count() == 8
assert bank_recon.filter("MatchStatus = 'UnmatchedBank'").count() == 7
gold["gold_bank_cash_reconciliation"] = bank_recon
credits = silver["vendor_statements"].filter("EntryType = 'CreditNote'").withColumn("DocumentType", F.lit("CreditNote"))
register("supplier_credits", credits)
gold["gold_supplier_credits"] = credits


## EN: 7. Budget and production summaries

BudgetActualUSD is the supplied budget model's controller-certified actual, not the imperfect staged ledger. Keep both measures visible. FULL OUTER join includes five no-budget unmapped accounts. VarianceUSD equals ControlUSD minus BudgetUSD; total -85,183.76 USD. Operations contain 184 records across S01 and S02, with no S03 production. Sum reported recovered ounces as decimal, preserving source precision; these are production, not sold ounces, revenue or cash. The conversion definition is ore tonnes * grade g/t * recovery fraction / 31.1034768. No gold-price time series, causal proof or supported cash runway is supplied.

## ID: 7. Ringkasan anggaran dan produksi

BudgetActualUSD adalah aktual tersertifikasi controller dari model anggaran yang disediakan, bukan ledger staging yang belum sempurna. Tampilkan kedua ukuran. FULL OUTER join mencakup lima akun tidak terpetakan tanpa anggaran. VarianceUSD sama dengan ControlUSD dikurangi BudgetUSD; total -85.183,76 USD. Operasi berisi 184 catatan S01 dan S02, tanpa produksi S03. Jumlahkan ons hasil pemulihan yang dilaporkan sebagai desimal, menjaga presisi sumber; ini produksi, bukan ons terjual, pendapatan atau kas. Definisi konversi adalah ton bijih * kadar g/t * fraksi pemulihan / 31,1034768. Tidak tersedia deret harga emas, bukti sebab-akibat atau proyeksi ketahanan kas yang didukung.


In [ ]:
budget_sql = """
SELECT COALESCE(b.Site,c.Site) AS Site, COALESCE(b.Period,c.Period) AS Period,
 COALESCE(b.Account,c.Account) AS Account, b.BudgetID,b.AccountName,b.DriverUnit,
 b.BudgetVolume,b.BudgetRateUSD,b.BudgetUSD,b.ActualUSD AS BudgetActualUSD,
 b.VarianceUSD AS SuppliedVarianceUSD,b.Assessment,b.ControlRef,
 c.KnownUSD_AllOccurrences,c.KnownUSD_ReplayExcluded,c.ControlUSD,
 c.ControlUSD-b.BudgetUSD AS VarianceUSD,
 b.ActualUSD-c.ControlUSD AS BudgetMinusControlUSD,
 c.ControlStatus,c.NullUSDRows,c.UnmappedAccountRows,c.RepeatedImportRows,
 CASE WHEN b.BudgetID IS NULL THEN 'NoBudget'
      WHEN NOT c.ControlPresent OR c.ControlPresent IS NULL THEN 'NoControl'
      WHEN b.ActualUSD <> c.ControlUSD THEN 'BudgetActualMismatch'
      ELSE 'BudgetActualAgreesWithControl' END AS BudgetStatus
FROM s_budget b FULL OUTER JOIN control_reconciliation c
 ON b.Site <=> c.Site AND b.Period <=> c.Period AND b.Account <=> c.Account
"""
budget = register("budget_summary", spark.sql(budget_sql))
assert budget.count() == 95
assert budget.agg(F.sum("BudgetUSD")).first()[0] == Decimal("26827033.83")
assert budget.agg(F.sum("VarianceUSD")).first()[0] == Decimal("-85183.76")
no_rows(budget.filter("BudgetMinusControlUSD <> 0"), "Budget actual/control disagreement")
gold["gold_budget_summary"] = budget
operations = spark.sql("""
SELECT Site, DATE_FORMAT(Date,'yyyy-MM') AS Period, COUNT(*) AS ProductionRows,
 SUM(PlannedOreTonnes) AS PlannedOreTonnes, SUM(ActualOreTonnes) AS ActualOreTonnes,
 SUM(ActualOreTonnes)-SUM(PlannedOreTonnes) AS OreTonnesVariance,
 SUM(GoldProducedOz) AS RecoveredGoldOz, SUM(PlannedGoldOz) AS PlannedGoldOz,
 SUM(GoldProducedOz)-SUM(PlannedGoldOz) AS GoldOzVariance,
 MIN(Date) AS FirstProductionDate, MAX(Date) AS LastProductionDate
FROM s_operations GROUP BY Site, DATE_FORMAT(Date,'yyyy-MM')
""")
register("operations_summary", operations)
assert operations.agg(F.sum("ProductionRows")).first()[0] == 184
gold["gold_operations_summary"] = operations


## EN: 8. Non-additive exception register

Union issue occurrences without dropping the underlying records. A ledger row and its control group can appear as different issues; AP candidates and receipt/price issues can overlap. Exception counts are not unique affected transaction counts, and ReviewAmount values must not be added as savings. Currency and AmountBasis explain each review amount. Null amounts remain unvalued. All 26 staged ledger exception occurrences must remain. Read-only analysis does not approve journals, funding, payment release or close signoff.

## ID: 8. Register pengecualian tidak aditif

Gabungkan kejadian isu tanpa menghapus catatan sumber. Baris ledger dan kelompok kontrolnya dapat muncul sebagai isu berbeda; kandidat AP serta isu penerimaan/harga dapat tumpang tindih. Jumlah pengecualian bukan jumlah transaksi terdampak unik, dan ReviewAmount tidak boleh dijumlahkan sebagai penghematan. Currency dan AmountBasis menjelaskan tiap nilai tinjauan. Nilai null tetap belum dinilai. Seluruh 26 kejadian pengecualian ledger staging harus tetap ada. Analisis baca-saja tidak menyetujui jurnal, pendanaan, pelepasan pembayaran atau persetujuan penutupan.


In [ ]:
exception_parts = []
exception_columns = ["ExceptionID", "Domain", "RecordID", "Site", "Period", "Account", "Currency", "IssueType", "ReviewAmount", "AmountBasis", "EvidenceRef"]

def exception_frame(frame, domain, record, issue, amount, currency, basis, evidence, site="Site", period="Period", account=None):
    selected = frame.select(
        F.concat_ws(":", F.lit(domain), record.cast("string"), issue).alias("ExceptionID"),
        F.lit(domain).alias("Domain"), record.cast("string").alias("RecordID"),
        F.col(site).cast("string").alias("Site") if site else F.lit(None).cast("string").alias("Site"),
        F.col(period).cast("string").alias("Period") if period else F.lit(None).cast("string").alias("Period"),
        F.col(account).cast("string").alias("Account") if account else F.lit(None).cast("string").alias("Account"),
        currency.alias("Currency"), issue.alias("IssueType"), amount.cast("decimal(28,2)").alias("ReviewAmount"),
        F.lit(basis).alias("AmountBasis"), evidence.cast("string").alias("EvidenceRef"))
    assert selected.columns == exception_columns
    exception_parts.append(selected)

exception_frame(ledger.filter("NormalizationStatus <> 'Ready'"), "Ledger", F.col("SourceRowID"),
    F.col("NormalizationStatus"), F.col("AmountUSD"), F.lit("USD"), "Known staged USD; null means unavailable",
    F.col("DocumentRef"), account="Account")
exception_frame(control.filter("ControlStatus <> 'Matched'"), "Control",
    F.concat_ws("|", F.coalesce("Site", F.lit("<NULL>")), F.coalesce("Period", F.lit("<NULL>")), F.coalesce("Account", F.lit("<NULL>"))),
    F.col("ControlStatus"), F.col("DifferenceUSD_AllOccurrences"), F.lit("USD"),
    "All occurrences minus controller certificate; not additive exposure",
    F.lit("ledger_control.csv"), account="Account")
for condition, issue, amount in [
    ("DuplicateCandidate", "DuplicateInvoiceCandidate", F.col("GrossUSD")),
    ("MissingReceipt", "MissingReceipt", F.col("NetAmount")),
    ("MissingPO", "MissingPO", F.col("NetAmount")),
    ("POVendorMismatch", "POVendorMismatch", F.col("NetAmount")),
    ("PriceDifferenceLocal <> 0", "PriceDifference", F.col("PriceDifferenceLocal")),
]:
    usd = issue == "DuplicateInvoiceCandidate"
    exception_frame(ap.filter(condition), "AP", F.col("InvoiceID"), F.lit(issue), amount,
        F.lit("USD") if usd else F.col("Currency"),
        "Gross candidate row USD including paid originals; non-additive, not incremental exposure" if usd else "Local currency review amount; not savings",
        F.col("POID"))
exception_frame(bank_recon.filter("MatchStatus <> 'Matched'"), "BankCash",
    F.concat_ws("|", "Ref", "Currency"), F.col("MatchStatus"),
    F.when(F.col("MatchStatus") == "UnmatchedBank", F.col("BankAllDates"))
     .when(F.col("MatchStatus") == "UnmatchedCash", F.col("CashAllDates")).otherwise(F.col("DifferenceAtClose")),
    F.col("Currency"), "Signed amount by Ref and Currency; timing is not an automatic journal",
    F.col("Ref"), site=None, period=None)
exception_frame(credits, "Supplier", F.col("StatementLineID"), F.lit("CreditNoteReview"),
    F.col("SignedAmount"), F.col("Currency"), "Credit note absent from AP invoice-only scope; validate recording elsewhere",
    F.col("EvidenceRef"), period=None)
exceptions = reduce(lambda a, b: a.unionByName(b), exception_parts)
unique(exceptions, ["ExceptionID"], "Exception identifier collision")
assert exceptions.filter("Domain = 'Ledger'").count() == 26
gold["gold_exception_register"] = exceptions


## EN: 9. Local table write and named CSV snapshot export

This cell writes nine managed gold Delta tables only in the attached demo Lakehouse. It does not publish a Data Agent or deploy anything outside the notebook. Each table is counted and schema-checked after storage. A bounded pandas export converts Spark columns to strings first, preventing Decimal-to-float conversion, and writes actual named UTF-8 CSV files under /lakehouse/default/Files/gold_export. Spark .csv() normally creates a folder of part files; this cell intentionally does not use that approach. export_manifest.json is written last with actual schemas, row counts, hashes and run identity. Writes across tables/files are not transactional. After any failure, do not use mixed snapshots; rerun successfully and compare manifest hashes. Download files from Lakehouse Files, then attach/upload the snapshot to Cowork using the available file control. This is not a live connector or an automatic SharePoint copy. No messages are sent. See Microsoft Learn: https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-notebook-load-data .

## ID: 9. Penulisan tabel lokal dan ekspor snapshot CSV bernama

Sel ini menulis sembilan tabel Delta gold terkelola hanya pada Lakehouse demo yang dilampirkan. Sel ini tidak menerbitkan Data Agent atau melakukan deployment di luar notebook. Jumlah baris dan skema tiap tabel diperiksa setelah penyimpanan. Ekspor pandas terbatas mengubah kolom Spark menjadi string terlebih dahulu, mencegah konversi Decimal ke float, lalu menulis file CSV UTF-8 bernama nyata di /lakehouse/default/Files/gold_export. Spark .csv() biasanya membuat folder bagian file; sel ini sengaja tidak memakai cara itu. export_manifest.json ditulis terakhir dengan skema aktual, jumlah baris, hash dan identitas proses. Penulisan lintas tabel/file tidak transaksional. Setelah kegagalan, jangan gunakan snapshot campuran; jalankan ulang sampai berhasil dan bandingkan hash manifest. Unduh file dari Lakehouse Files, lalu lampirkan/unggah snapshot ke Cowork melalui kontrol file yang tersedia. Ini bukan konektor langsung atau salinan SharePoint otomatis. Tidak ada pesan dikirim. Lihat Microsoft Learn: https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-notebook-load-data .


In [ ]:
metadata_records = []
for filename, spec in contract["csv_files"].items():
    metadata_records.append((
        filename, int(spec["rows"]), spec["sha256"], SOURCE_AS_OF, CLOSE_CUTOFF,
        contract["source_ledger_scope"], contract["control_generation_assumption"],
        "Synthetic snapshot; not a live connector; not audited", RUN_UTC))
metadata_schema = T.StructType([
    T.StructField("SourceFile", T.StringType(), False), T.StructField("InputRows", T.LongType(), False),
    *[T.StructField(name, T.StringType(), False) for name in [
        "SHA256", "SourceAsOfDate", "CloseCutoffDate", "LedgerScope", "ControlAssumption", "SnapshotStatus", "RunUTC"]]
])
gold["gold_source_metadata"] = spark.createDataFrame(metadata_records, metadata_schema)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_manifest = {"run_utc": RUN_UTC, "source_as_of_date": SOURCE_AS_OF, "close_cutoff_date": CLOSE_CUTOFF, "exports": {}}
for name, frame in gold.items():
    if name != "gold_source_metadata":
        frame = frame.withColumn("SourceAsOfDate", F.lit(SOURCE_AS_OF)).withColumn("CloseCutoffDate", F.lit(CLOSE_CUTOFF)).withColumn("RunUTC", F.lit(RUN_UTC))
    assert set(frame.columns) == set(GOLD_COLUMNS[name]), f"Gold query contract mismatch: {name}"
    frame = frame.select(*GOLD_COLUMNS[name])
    assert not any(isinstance(field.dataType, (T.DoubleType, T.FloatType)) for field in frame.schema.fields), name
    count = frame.count()
    assert count <= MAX_EXPORT_ROWS, f"Export too large for driver: {name}"
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    stored = spark.table(name)
    assert stored.count() == count
    assert stored.schema.simpleString() == frame.schema.simpleString()
    gold[name] = stored
    # Pandas receives strings, so Decimal money cannot become a binary float.
    text_frame = stored.select(*[F.col(c).cast("string").alias(c) for c in stored.columns])
    pdf = text_frame.toPandas().fillna("")
    assert len(pdf) == count
    target = EXPORT_DIR / (name + ".csv")
    pdf.to_csv(target, index=False, encoding="utf-8", lineterminator="\n")
    with target.open(encoding="utf-8", newline="") as stream:
        check = csv.reader(stream)
        assert next(check) == stored.columns
        assert sum(1 for _ in check) == count
    export_manifest["exports"][target.name] = {
        "rows": count, "columns": stored.columns, "spark_schema": stored.schema.jsonValue(),
        "sha256": hashlib.sha256(target.read_bytes()).hexdigest()}
# The manifest is written last. A failed run can leave mixed snapshots: rerun
# successfully and verify hashes before using any exported file.
(EXPORT_DIR / "export_manifest.json").write_text(json.dumps(export_manifest, indent=2), encoding="utf-8")
print("EN: Complete. Review controls, download named CSV files from Lakehouse Files/gold_export, then attach the snapshot to Cowork. Nothing was sent.")
print("ID: Selesai. Tinjau kontrol, unduh CSV bernama dari Lakehouse Files/gold_export, lalu lampirkan snapshot ke Cowork. Tidak ada yang dikirim.")
display(gold["gold_control_reconciliation"].orderBy("Site", "Period", "Account"))
